# Managing software environments

## Learning objectives

- Declare a package in `environment.yml`, and lock the environment with conda.
- Declare the same package in `pyproject.toml`, and lock it with uv.
- Tell an environment file from a lock: what you asked for, and what you got.
- See why a conda lock is one file per platform, and a uv lock is one file for every platform.

## The problem

- The workshop's exercises use one project, a study of Palmer penguins, and add a capability in each exercise.
- `project` has a short Python script, `analyze.py`. It reads the 2007 field season and computes male minus female mean body mass for each species, in grams.
- `python analyze.py` fails. It imports pandas, and the project's dependency files don't declare it.
- The project has two dependency files: `environment.yml` for conda and `pyproject.toml` for uv. You fix the same problem with each tool.

## What you'll do

1. Step 1, conda: declare pandas, build the environment, write the lock and rebuild from the lock.
2. Step 2, uv: declare pandas, write the lock and rebuild from the lock.
3. Run one check for both steps.

- The whole exercise takes about 20 minutes.
- Only reading? Each step shows the failure and how to approach the fix.

## The rule

- The document and the data are the pets. Everything that runs them is cattle.
- Here the cattle are the conda environments and `.venv`. The environment file and its lock are in Git, so anyone can build them again.

## Get a terminal

- You read this page and paste commands into a terminal. Nothing runs on the page.
- The output shown here is recorded from the untouched project, so it stops at the failures. The fixes are yours.

**[Open it in Binder](https://mybinder.org/v2/gh/ohpcsim/reproducibility-in-ai/binder-01-environments?urlpath=lab/workspaces/exercise)**

- Free, with nothing to install. The first start can take a few minutes.
- The terminal is on the left, already in the `project` folder. This page is on the right.
- Copy a command: select it and press Ctrl+C, or Cmd+C on a Mac.
- Paste it in the terminal: Ctrl+Shift+V, or right-click and choose Paste. Cmd+V on a Mac is untested, so right-click if it doesn't work.
- Edit a file: double-click it in the file panel on the far left. Save with Ctrl+S.
- A session is temporary and has 2 GB of memory. Download anything you want to keep.

**Or use your own terminal**

- Linux or macOS, on x86_64 or ARM, or Windows through WSL2. On Windows, keep the folder in your WSL2 home, not under `/mnt/c`.
- You need [conda](https://github.com/conda-forge/miniforge). The exercise's environment includes uv, so you don't install it separately. You don't need Jupyter.
- Paste this once:

```bash
git clone --branch binder-01-environments --single-branch https://github.com/ohpcsim/reproducibility-in-ai.git environments
cd environments
conda env create --quiet --file .binder/environment.yml
conda activate repro-environments
python scripts/setup_project.py
cd project
```

- Every command below runs in that `project` folder.
- To start over, go to the folder above it and run `python scripts/setup_project.py --reset`. This deletes your work.

## The project

- `project` is a Git repository with one commit: `analyze.py`, `data/penguins.csv` and the two dependency files.
- `environment.yml` lists the packages for conda. `pyproject.toml` lists them for uv. Neither lists pandas.

```bash
git log --format='%h  %an: %s'
cat environment.yml
```

In [1]:
# Recorded output. You don't need to run this.
import sys

sys.path.insert(0, ".exercise")
from page import run

run("git log --format='%h  %an: %s'", "cat environment.yml")

$ git log --format='%h  %an: %s'
106ad02  Priya Rao: Add the penguin analysis

$ cat environment.yml
name: penguins
channels:
  - conda-forge
dependencies:
  - python=3.12


## Step 1: declare and lock with conda

**Problem:** the README says to run `python analyze.py`, and it fails.

```bash
python analyze.py
```

In [2]:
# Recorded output. You don't need to run this.
run("python analyze.py")

$ python analyze.py
Traceback (most recent call last):
  File "~/project/analyze.py", line 4, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'


**Facts**

- `analyze.py` imports pandas, and `environment.yml` doesn't list it. It runs only where someone installed pandas by hand.
- Add a package to `environment.yml` when you start using it.
- `environment.yml` lists what you asked for. Leave the versions out, so conda picks current ones that work together.
- The lock lists what you got: the exact file of every package conda installed, with its checksum.
- `conda list --explicit --md5` writes that lock.
- Each URL in the lock includes a platform folder, such as `linux-64` or `osx-arm64`. conda packages are built for one platform, so a lock from Linux doesn't install on a Mac.
- That's why the lock's name includes the platform. `conda config --show subdir` prints the platform's name.
- `conda create --file` builds an environment from the lock, with exactly those packages and no solving.

**Steps**

1. Open `environment.yml` in the `project` folder. Add `  - pandas` on a new line, in the same form as the others, with no version. Save the file.
2. Confirm the edit. This should show the one line you added. If it prints nothing, you edited another file or didn't save.

```bash
git diff environment.yml
```

3. Build the environment from that file, and write the lock. This takes a minute or two.

```bash
conda env create --quiet --prefix ../envs/conda --file environment.yml
conda list --prefix ../envs/conda --explicit --md5 > explicit-$(conda config --show subdir | cut -d' ' -f2).txt
```

4. Look at the lock. The header names the platform, and the pandas line is a URL under that platform's folder.

```bash
head -4 explicit-*.txt
grep /pandas- explicit-*.txt
```

- On Linux on x86_64, the lock is `explicit-linux-64.txt`, and the pandas line looks like this:

```text
https://conda.anaconda.org/conda-forge/linux-64/pandas-3.1.1-np2py312h91ec553_0.conda#d3c0f076ef954125c9c3db84794bc51b
```

- On a Mac with Apple silicon, the lock is `explicit-osx-arm64.txt`:

```text
https://conda.anaconda.org/conda-forge/osx-arm64/pandas-3.1.1-np2py312ha2f16c6_0.conda#ee11e486200da574175c16e7828ac72d
```

5. Commit both files. The commit should say `2 files changed`.

```bash
git add environment.yml explicit-*.txt
git commit -m "Declare pandas and lock the conda environment"
```

6. Build a second environment from the lock, and run the analysis in it.

```bash
conda create --yes --quiet --prefix ../envs/conda-lock --file explicit-$(conda config --show subdir | cut -d' ' -f2).txt
../envs/conda-lock/bin/python analyze.py
```

## Step 2: declare and lock with uv

**Problem:** `pyproject.toml` doesn't list pandas either.

```bash
cat pyproject.toml
```

In [3]:
# Recorded output. You don't need to run this.
run("cat pyproject.toml")

$ cat pyproject.toml
[project]
name = "penguins"
version = "0.1.0"
requires-python = ">=3.12,<3.13"
dependencies = []


**Facts**

- `pyproject.toml` is the standard file for a Python project. Its `dependencies` list is what you asked for.
- `uv add pandas` adds pandas to that list, writes the lock to `uv.lock` and installs everything into `.venv`.
- uv writes a lower bound such as `pandas>=3.1.1`. The exact version goes in the lock.
- PyPI publishes a separate wheel for each platform. `uv.lock` lists them all, so one lock covers Linux, macOS and Windows, on x86_64 and ARM.
- When it installs, uv picks the wheel for your platform from the lock.
- `uv sync --locked` builds `.venv` from the lock. It stops with an error if the lock doesn't match `pyproject.toml`.
- `.venv` is in `.gitignore`. It's cattle.

**Steps**

1. Declare pandas, and confirm the change.

```bash
uv add pandas
git diff pyproject.toml
```

2. Look at the lock. It lists a pandas wheel for each platform.

```bash
grep -o 'pandas-[^"]*\.whl' uv.lock
```

- The list looks like this:

```text
pandas-3.1.1-cp312-cp312-macosx_10_13_x86_64.whl
pandas-3.1.1-cp312-cp312-macosx_11_0_arm64.whl
pandas-3.1.1-cp312-cp312-manylinux_2_24_aarch64.manylinux_2_28_aarch64.whl
pandas-3.1.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
pandas-3.1.1-cp312-cp312-musllinux_1_2_aarch64.whl
pandas-3.1.1-cp312-cp312-musllinux_1_2_x86_64.whl
pandas-3.1.1-cp312-cp312-pyemscripten_2024_0_wasm32.whl
pandas-3.1.1-cp312-cp312-win_amd64.whl
pandas-3.1.1-cp312-cp312-win_arm64.whl
```

3. Commit both files. The commit should say `2 files changed`.

```bash
git add pyproject.toml uv.lock
git commit -m "Declare pandas and lock the uv environment"
```

4. Delete `.venv`, build it again from the lock, and run the analysis.

```bash
rm -rf .venv
uv sync --locked
uv run python analyze.py
```

## Check your work

- Run this when both steps are done. Each line says what to do next.
- It clones your last commit into a throwaway folder, so it only sees what you committed.
- It builds an environment from each lock and runs the analysis in it. It takes under a minute.
- If you did only one step, it says which one is left.

```bash
python ../check.py
```

On the starter project it fails like this:

In [4]:
# Recorded output. You don't need to run this.
run("python ../check.py")

$ python ../check.py
Step 1, conda:
FAIL  The committed environment.yml doesn't declare pandas. Add it as '  - pandas', with no version. Step 1.
FAIL  No explicit-linux-aarch64.txt is committed. Write it with conda list --explicit --md5, and commit it. Step 1.
FAIL  Not tested: an environment built from explicit-linux-aarch64.txt runs analyze.py. Fix the lock first.
Step 2, uv:
FAIL  The committed pyproject.toml doesn't declare pandas. Run uv add pandas. Step 2.
FAIL  No uv.lock is committed. uv add writes it. Commit it with pyproject.toml. Step 2.
FAIL  Not tested: uv sync --locked builds an environment that runs analyze.py. Fix uv.lock first.

0 of 6 parts pass.


## conda and uv

- An environment file, `environment.yml` or `pyproject.toml`, is what you asked for. A lock, `explicit-<platform>.txt` or `uv.lock`, is what you got.
- Commit both. The environment file says what you meant, and the lock rebuilds what you ran.
- conda installs packages that aren't Python, such as R, C and C++ libraries, CUDA and MPI.
- A conda explicit lock covers one platform and architecture. Each platform you support needs its own file.
- uv installs Python packages from PyPI only.
- A uv lock covers every platform in one file.
- [conda-lock](https://conda.github.io/conda-lock/) and [pixi](https://pixi.prefix.dev/latest/workspace/lock_file/) solve for several platforms and write one lock file for conda packages.

## Example lock files from four machines

- These are four locks of the same `environment.yml`, with pandas added, written on November 8, 2026, between 18:49:50 and 18:50:13 UTC:
  - [Linux on x86_64, Windows WSL2](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/explicit-linux-64-wsl2.txt)
  - [Linux on x86_64, Binder](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/explicit-linux-64-binder.txt)
  - [Linux on ARM, NVIDIA DGX Spark](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/explicit-linux-aarch64-spark.txt)
  - [macOS on Apple silicon](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/explicit-osx-arm64-mac.txt)
  - [uv.lock](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/uv.lock), the same file on all four machines
- Every machine got the same versions, such as pandas 3.1.1. The build strings and platform folders differ, so each conda lock installs only on its own platform.
- The Linux locks have 38 packages and the Mac lock has 33. Some libraries exist on one platform only: Linux gets `libstdcxx` and `libgomp`, and the Mac gets `libcxx` and `llvm-openmp`.
- The WSL2 and Binder locks list the same 38 packages with the same checksums. Only the header line naming the conda version differs.
- They match because both used only conda-forge, solved within seconds of each other, and report the same glibc, 2.39. A solve on another day can pick newer packages. Commit the lock, not only the environment file.
- The [README](https://github.com/ohpcsim/reproducibility-in-ai/blob/main/exercises/01-environments/examples/locks/README.md) lists each machine, its conda version and the solve times.

## Summary

- Add a package to the environment file when you start using it.
- Write the lock and commit it with the environment file.
- Rebuild from the lock, and run the analysis, before you trust it.

## On your own project

- Run your analysis in a new environment built only from your environment file. Anything missing shows up as an import error.
- Write the lock today, for each platform you use.
- If everything comes from PyPI, uv gives you one lock for every platform. If you need packages that aren't Python, use conda, and lock each platform.
- Optional, on your own machine: [1b. Environments in containers](containers.md) builds both locks into Docker, Podman and Apptainer images.

## About the data

- Palmer penguins, CC0 1.0, from the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) package. `data/penguins.csv` is its 2007 rows, 110 penguins, unchanged.
- Data: palmerpenguins (Horst, Hill and Gorman 2020, doi:10.5281/zenodo.3960218), CC0, from Gorman, Williams and Fraser (2014), PLOS ONE 9(3): e90081. Collected by Palmer Station LTER with NSF support.